# 02 — Finance & Safety Agents
## Agentic X | Studio 67

This notebook develops and evaluates two independent agents:

- Finance Agent: Evaluates budgets, costs, and financial feasibility.
- Safety Agent: Evaluates risks, permits, and safety constraints.

Both agents use Python-based decision-making and return structured evaluations for multi-agent negotiation.

In [1]:
from pathlib import Path
import subprocess
import sys
import os

REPO_URL = "https://github.com/ProjectsSpaceHub/AgenticX.git"
ROOT = Path("/content/AgenticX")

if not (ROOT / "core" / "environment.py").exists():
    subprocess.run(
        ["git", "clone", REPO_URL, str(ROOT)],
        check=True
    )

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "pandas", "openpyxl", "pydantic"
    ],
    check=True
)

print("Environment setup completed successfully!")

Environment setup completed successfully!


#Cell 3 — Code: Import Libraries

In [2]:
import pandas as pd
from IPython.display import display

from core.environment import load_dataset, get_case
from agents.finance import FinanceAgent
from agents.safety import SafetyAgent
from core.models import AgentEvaluation

print("All libraries imported successfully!")

All libraries imported successfully!


#Cell 4 — Code: Load Dataset

In [3]:
cases_df, options_df = load_dataset()

print("Total Cases:", len(cases_df))
print("Total Options:", len(options_df))

assert len(cases_df) == 150
assert len(options_df) == 450

print("Dataset validation passed!")

display(cases_df.head())

Total Cases: 150
Total Options: 450
Dataset validation passed!


,case_id,scenario,incident,budget_sar,time_left_min,severity,backup_available,crew_available,venue_access,notes,...,initial_budget_sar,committed_spend_sar,incident_type,required_quality_min,safety_policy,case_profile,crew_count_available,unplanned_charge_sar,projected_total_spend_sar,projected_overrun_sar
0,SCR-001,Screen Failure,Main LED screen failed during final rehearsal,8500,100,Critical,True,True,True,Synthetic test input; no precomputed agent dec...,...,8500,0,display_failure,60,Reject non-compliant options; escalate if none...,Baseline feasible,3,0,0,0
1,SCR-002,Screen Failure,LED wall lost signal before keynote,2400,60,High,False,True,True,Synthetic test input; no precomputed agent dec...,...,2400,0,display_failure,50,Reject non-compliant options; escalate if none...,Tight budget feasible,2,0,0,0
2,SCR-003,Screen Failure,Display controller failure before exhibition o...,7220,90,High,True,True,True,Synthetic test input; no precomputed agent dec...,...,7220,0,display_failure,50,Reject non-compliant options; escalate if none...,Time pressure feasible,2,0,0,0
3,SCR-004,Screen Failure,Screen power module stopped working,14000,150,Medium,False,True,True,Synthetic test input; no precomputed agent dec...,...,14000,0,display_failure,50,Reject non-compliant options; escalate if none...,Resource-constrained feasible,3,0,0,0
4,SCR-005,Screen Failure,Main stage screen shows intermittent blackout,11000,150,Critical,True,True,True,Synthetic test input; no precomputed agent dec...,...,11000,0,display_failure,55,Reject non-compliant options; escalate if none...,Multi-option feasible,1,0,0,0


#Cell 5 — Finance & Safety Initialization

In [4]:
finance_agent = FinanceAgent()
safety_agent = SafetyAgent()

print("Finance Agent initialized successfully!")
print("Safety Agent initialized successfully!")

Finance Agent initialized successfully!
Safety Agent initialized successfully!


#Cell 6 — Test Finance & Safety on Screen Failure

In [5]:
case, options = get_case("SCR-001")

evaluations = []

for option in options:
    finance_result = finance_agent.evaluate(case, option)
    safety_result = safety_agent.evaluate(case, option)

    evaluations.append(finance_result.model_dump())
    evaluations.append(safety_result.model_dump())

results_df = pd.DataFrame(evaluations)

display(results_df)

,agent,option_id,approved,score,reason,blocking
0,Finance,SCR-001-O1,True,100.0,Committed plus option cost: 1290.00 SAR versus...,True
1,Safety,SCR-001-O1,True,100.0,All safety checks passed,True
2,Finance,SCR-001-O2,True,100.0,Committed plus option cost: 5920.00 SAR versus...,True
3,Safety,SCR-001-O2,True,100.0,All safety checks passed,True
4,Finance,SCR-001-O3,True,100.0,Committed plus option cost: 1210.00 SAR versus...,True
5,Safety,SCR-001-O3,True,100.0,All safety checks passed,True


In [6]:
print(results_df.to_string(index=False))

  agent  option_id  approved  score                                                             reason  blocking
Finance SCR-001-O1      True  100.0 Committed plus option cost: 1290.00 SAR versus budget 8500.00 SAR.      True
 Safety SCR-001-O1      True  100.0                                           All safety checks passed      True
Finance SCR-001-O2      True  100.0 Committed plus option cost: 5920.00 SAR versus budget 8500.00 SAR.      True
 Safety SCR-001-O2      True  100.0                                           All safety checks passed      True
Finance SCR-001-O3      True  100.0 Committed plus option cost: 1210.00 SAR versus budget 8500.00 SAR.      True
 Safety SCR-001-O3      True  100.0                                           All safety checks passed      True


#Cell 7 — Validate Agent Outputs

In [7]:
assert len(results_df) == 6

assert set(results_df["agent"]) == {"Finance", "Safety"}

assert results_df["score"].between(0, 100).all()

assert results_df["reason"].notna().all()

assert results_df["blocking"].all()

print("Finance and Safety validation passed!")

Finance and Safety validation passed!


#Cell 8 — Test Budget Overrun & Safety Conflict

In [8]:
test_cases = ["BDG-001", "SFT-001"]

all_results = []

for case_id in test_cases:

    case, options = get_case(case_id)

    print(f"\nTesting: {case_id}")

    for option in options:

        finance_result = finance_agent.evaluate(case, option)
        safety_result = safety_agent.evaluate(case, option)

        all_results.append({
            "case_id": case_id,
            "option_id": option["option_id"],
            "finance_approved": finance_result.approved,
            "finance_reason": finance_result.reason,
            "safety_approved": safety_result.approved,
            "safety_reason": safety_result.reason
        })

test_results_df = pd.DataFrame(all_results)

print(test_results_df.to_string(index=False))


Testing: BDG-001

Testing: SFT-001
case_id  option_id  finance_approved                                                                                  finance_reason  safety_approved                                                                    safety_reason
BDG-001 BDG-001-O1              True  Projected spend after action: 20836.00 SAR versus budget 21086.00 SAR; net saving 1450.00 SAR.             True                                                         All safety checks passed
BDG-001 BDG-001-O2              True  Projected spend after action: 20686.00 SAR versus budget 21086.00 SAR; net saving 1600.00 SAR.             True                                                         All safety checks passed
BDG-001 BDG-001-O3             False Projected spend after action: 29176.00 SAR versus budget 21086.00 SAR; net saving -6890.00 SAR.             True                                                         All safety checks passed
SFT-001 SFT-001-O1              True    

#Cell 9 — Automated Decision Tests

In [9]:
# Test Finance Agent decisions

finance_case = {
    "scenario": "Budget Overrun",
    "initial_budget_sar": 10000,
    "projected_total_spend_sar": 12000
}

affordable_option = {
    "option_id": "TEST-F1",
    "cost_sar": 200,
    "gross_savings_sar": 2500
}

expensive_option = {
    "option_id": "TEST-F2",
    "cost_sar": 200,
    "gross_savings_sar": 1000
}

assert finance_agent.evaluate(
    finance_case, affordable_option
).approved is True

assert finance_agent.evaluate(
    finance_case, expensive_option
).approved is False

# Test Safety Agent decisions

safe_option = {
    "option_id": "TEST-S1",
    "safety_ok": True,
    "permit_ok": True,
    "mitigation_verified": True,
    "risk_score_after_mitigation": 2
}

assert safety_agent.evaluate(
    {}, safe_option
).approved is True

for field in [
    "safety_ok",
    "permit_ok",
    "mitigation_verified"
]:
    unsafe_option = safe_option.copy()
    unsafe_option[field] = False

    assert safety_agent.evaluate(
        {}, unsafe_option
    ).approved is False

high_risk_option = safe_option.copy()
high_risk_option["risk_score_after_mitigation"] = 8

assert safety_agent.evaluate(
    {}, high_risk_option
).approved is False

print("All automated decision tests passed!")

All automated decision tests passed!


In [17]:
import subprocess
import importlib
from pathlib import Path

# Get latest Safety Agent from GitHub
repo = Path("/content/AgenticX")

subprocess.run(
    ["git", "-C", str(repo), "fetch", "origin", "main"],
    check=True
)

updated_code = subprocess.check_output(
    [
        "git", "-C", str(repo),
        "show", "FETCH_HEAD:agents/safety.py"
    ],
    text=True
)

(repo / "agents/safety.py").write_text(
    updated_code,
    encoding="utf-8"
)

# Reload the updated agent
import agents.safety as safety_module

importlib.reload(safety_module)

safety_agent = safety_module.SafetyAgent()

# Test zero-risk option
zero_risk_option = {
    "option_id": "TEST-ZERO-RISK",
    "safety_ok": True,
    "permit_ok": True,
    "mitigation_verified": True,
    "risk_score_after_mitigation": 0
}

result = safety_agent.evaluate({}, zero_risk_option)

assert result.approved is True

print("Zero-risk safety test passed!")

Zero-risk safety test passed!


#Cell 10 — Evaluate All 150 Cases

In [18]:
from core.environment import (
    clean,
    HIDDEN_OPTION_FIELDS,
    HIDDEN_CASE_FIELDS
)

batch_results = []

# Organize options once to avoid repeated Excel loading
options_by_case = {
    case_id: group.to_dict("records")
    for case_id, group in options_df.groupby("case_id")
}

for row in cases_df.to_dict("records"):

    case = {
        k: clean(v)
        for k, v in row.items()
        if k not in HIDDEN_CASE_FIELDS
    }

    case_id = case["case_id"]

    for raw_option in options_by_case[case_id]:

        option = {
            k: clean(v)
            for k, v in raw_option.items()
            if k not in HIDDEN_OPTION_FIELDS
        }

        finance_result = finance_agent.evaluate(case, option)
        safety_result = safety_agent.evaluate(case, option)

        batch_results.append({
            "case_id": case_id,
            "scenario": case["scenario"],
            "option_id": option["option_id"],
            "finance_approved": finance_result.approved,
            "safety_approved": safety_result.approved,
            "both_approved": (
                finance_result.approved
                and safety_result.approved
            ),
            "finance_reason": finance_result.reason,
            "safety_reason": safety_result.reason
        })

batch_df = pd.DataFrame(batch_results)

assert len(batch_df) == 450
assert batch_df["case_id"].nunique() == 150
assert not batch_df["option_id"].duplicated().any()

print("Total cases tested:", batch_df["case_id"].nunique())
print("Total options evaluated:", len(batch_df))
print("Finance approvals:", batch_df["finance_approved"].sum())
print("Safety approvals:", batch_df["safety_approved"].sum())

display(batch_df.head(10))

Total cases tested: 150
Total options evaluated: 450
Finance approvals: 357
Safety approvals: 355


,case_id,scenario,option_id,finance_approved,safety_approved,both_approved,finance_reason,safety_reason
0,SCR-001,Screen Failure,SCR-001-O1,True,True,True,Committed plus option cost: 1290.00 SAR versus...,All safety checks passed
1,SCR-001,Screen Failure,SCR-001-O2,True,True,True,Committed plus option cost: 5920.00 SAR versus...,All safety checks passed
2,SCR-001,Screen Failure,SCR-001-O3,True,True,True,Committed plus option cost: 1210.00 SAR versus...,All safety checks passed
3,SCR-002,Screen Failure,SCR-002-O1,True,True,True,Committed plus option cost: 1230.00 SAR versus...,All safety checks passed
4,SCR-002,Screen Failure,SCR-002-O2,False,True,False,Committed plus option cost: 6300.00 SAR versus...,All safety checks passed
5,SCR-002,Screen Failure,SCR-002-O3,True,True,True,Committed plus option cost: 1230.00 SAR versus...,All safety checks passed
6,SCR-003,Screen Failure,SCR-003-O1,True,True,True,Committed plus option cost: 1180.00 SAR versus...,All safety checks passed
7,SCR-003,Screen Failure,SCR-003-O2,True,True,True,Committed plus option cost: 6620.00 SAR versus...,All safety checks passed
8,SCR-003,Screen Failure,SCR-003-O3,True,True,True,Committed plus option cost: 1050.00 SAR versus...,All safety checks passed
9,SCR-004,Screen Failure,SCR-004-O1,True,True,True,Committed plus option cost: 1220.00 SAR versus...,All safety checks passed


#Cell 11 — Performance Summary

In [19]:
summary_df = batch_df.groupby("scenario").agg(
    total_options=("option_id", "count"),
    finance_approved=("finance_approved", "sum"),
    safety_approved=("safety_approved", "sum"),
    both_approved=("both_approved", "sum")
).reset_index()

summary_df["finance_approval_rate"] = (
    summary_df["finance_approved"]
    / summary_df["total_options"] * 100
).round(2)

summary_df["safety_approval_rate"] = (
    summary_df["safety_approved"]
    / summary_df["total_options"] * 100
).round(2)

display(summary_df)

print("Evaluation summary generated successfully")


,scenario,total_options,finance_approved,safety_approved,both_approved,finance_approval_rate,safety_approval_rate
0,Budget Overrun,150,100,133,91,66.67,88.67
1,Safety Conflict,150,130,92,77,86.67,61.33
2,Screen Failure,150,127,130,109,84.67,86.67


Evaluation summary generated successfully


#Cell 12 — Finance & Safety PEAS

In [20]:
peas_data = [
    {
        "Agent": "Finance",
        "Performance": "Budget compliance and financial feasibility",
        "Environment": "Studio 67 event crisis management",
        "Actuators": "Approve, reject, and explain financial decisions",
        "Sensors": "Budget, costs, expected savings, projected spending",
        "Type": "Deliberative (rule-based)",
        "Justification": "Evaluates financial consequences before deciding"
    },
    {
        "Agent": "Safety",
        "Performance": "Safety compliance and risk prevention",
        "Environment": "Studio 67 event crisis management",
        "Actuators": "Approve, reject, and issue safety veto",
        "Sensors": "Risk score, permits, safety and mitigation status",
        "Type": "Reactive (rule-based)",
        "Justification": "Immediately rejects options violating safety constraints"
    }
]

peas_df = pd.DataFrame(peas_data)

display(peas_df)

,Agent,Performance,Environment,Actuators,Sensors,Type,Justification
0,Finance,Budget compliance and financial feasibility,Studio 67 event crisis management,"Approve, reject, and explain financial decisions","Budget, costs, expected savings, projected spe...",Deliberative (rule-based),Evaluates financial consequences before deciding
1,Safety,Safety compliance and risk prevention,Studio 67 event crisis management,"Approve, reject, and issue safety veto","Risk score, permits, safety and mitigation status",Reactive (rule-based),Immediately rejects options violating safety c...


#Cell 13 — Save Evaluation Results

In [21]:
from pathlib import Path

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

batch_df.to_csv(
    results_dir / "finance_safety_evaluations.csv",
    index=False
)

summary_df.to_csv(
    results_dir / "finance_safety_summary.csv",
    index=False
)

print("Finance and Safety results saved successfully")

Finance and Safety results saved successfully


#Cell 14 — Conclusion

## Conclusion

This notebook implemented and evaluated two independent
agents: Finance and Safety.

The Finance Agent assesses financial feasibility, including
budget limits, implementation costs, and projected savings.

The Safety Agent enforces mandatory safety constraints
and rejects unsafe alternatives.

Both agents produce structured AgentEvaluation outputs
for communication and negotiation.

Their decisions were tested across 150 synthetic cases
and 450 candidate solutions.

Final multi-agent decisions require Creative, Operations,
and the Negotiation Engine.